# Report (draft)
Feel free to edit this!

## Part A - File Generation

In [ ]:
import os

sizes = [8,64,512,4096,32768,262144,2097152]

for s in sizes:
    with open(f"file_{s}.bin","wb") as f:
        f.write(os.urandom(s))

This code was used to generate binary files of the requested byte sizes. Binary files were used so the input consists of uniformly random byte sequences, avoiding any encoding-related bias that could arise from text-based files.

## Part D - SHA-256 Measurements

In [ ]:
import hashlib
import timeit
# import statistics

def sha256_file(filepath, runs=100):
    with open(filepath, "rb") as f:
        data = f.read()

    timer = timeit.Timer(lambda: hashlib.sha256(data).digest())

    # Warm-up
    timer.timeit(number=50)

    raw_times = timer.repeat(repeat=runs, number=10)

    return [(t / 10) * 1000000 for t in raw_times]

The function reads the entire file into memory and measures the execution time of the SHA-256 hashing operation on its data. We perform a preliminary warm-up execution using the timing framework to reduce the influence of interpreter initialization effects, CPU caching, and other runtime overheads.

The `timeit.Timer` object is then used to execute the hashing operation multiple times, each being 10 repeated hashes per measurement. The function returns a list of execution times corresponding to a single SHA-256 computation, in microseconds.

This approach isolates the computational cost of the cryptographic hash function from file input/output overhead.

In [ ]:
import sha_crypto
import statistics
import matplotlib.pyplot as plt
import pandas as pd
import math

results = []

print("File Size (B) | Throughput (B/µs) | CI ")
print("--------------+-------------------+--------")

files = [
    (8, "file_8.bin"),
    (64, "file_64.bin"),
    (512, "file_512.bin"),
    (4096, "file_4096.bin"),
    (32768, "file_32768.bin"),
    (262144, "file_262144.bin"),
    (2097152, "file_2097152.bin")
]

for size, f in files:

    runs = 100 if size > 512 else 500

    all_times = sha_crypto.sha256_file(f, runs=runs)

    avg_time = statistics.mean(all_times)
    std_dev = statistics.stdev(all_times)

    throughput = size / avg_time

    n = len(all_times)
    ci = 1.96 * (std_dev / math.sqrt(n))

    ci_rel = ci / avg_time

    print(f"{size:<13} | {throughput:<17.6f} | ±{ci_rel:.4f}")

    results.append({
        "Size": size,
        "MeanTime": avg_time,
        "CI": ci,
    })

This section evaluates SHA-256 performance across different file sizes by repeatedly hashing each file using `sha_crypto.sha256_file()` and collecting execution time samples for each input size.

For file sizes above 512 bytes, 100 independent runs are performed, while for smaller file sizes (8, 64, and 512 bytes), 500 runs are used. This is because the execution times are so short that they fall close to the resolution limit of the timing method. At this scale, fixed overheads like function calls, interpreter execution, and CPU scheduling have a noticeable impact on the results, which leads to higher variance relative to the mean, which means that more samples are needed to get a stable average.

As such, these measurements aren't really representative of SHA-256's asymptotic performance, but they're still useful for seeing where execution shifts from being overhead-dominated (small file sizes) to compute-dominated (larger file sizes). Because of this, results for small file sizes should be taken as rough indicators rather than direct comparisons to the larger input results.

For each file size, the following metrics are calculated:

- **Mean execution time:** average cost of computing the SHA-256 hash for a given file size, used as the primary performance indicator.
- **Standard deviation:** variability across repeated measurements caused by system-level noise such as CPU scheduling and caching effects.
- ***95%* confidence interval:** estimate of the uncertainty in the mean execution time, allowing comparison of whether differences between file sizes are statistically meaningful.
- **Throughput:** represents the processing rate in bytes per microsecond, derived from the ratio of input size to mean execution time, and is used to evaluate scalability.

These metrics allow performance to be interpreted both in terms of absolute execution time and normalized throughput, enabling analysis of how SHA-256 scales with increasing input size.

Finally, the results are stored and passed to a visualization stage using matplotlib and pandas to generate plots of execution time versus input size. The plotting code used to generate the visualizations was assisted by an **AI tool (Google Gemini)**. The implementation was reviewed by us to ensure correctness and alignment with the experimental requirements.

### Example Output Table

| File Size (B) | Throughput (B/µs) | 95% CI (µs) |
|---------------|-------------------|-------------|
| 8             | 25.399438         | ±0.0301     |
| 64            | 194.617963        | ±0.0100     |
| 512           | 945.414848        | ±0.0409     |
| 4096          | 1765.553003       | ±0.0203     |
| 32768         | 2087.494349       | ±0.0083     |
| 262144        | 2219.469823       | ±0.0080     |
| 2097152       | 2183.131356       | ±0.0061     |

This table shows SHA-256 throughput (bytes processed per microsecond) and how uncertain each measurement is. For very small inputs, throughput is much lower because fixed overheads (like function calls and interpreter execution) make up a large portion of the total execution time, as discussed above.

As file size increases, those overheads become less significant and throughput rises quickly. Past around 32 KB it levels off, which is where the actual SHA-256 computation starts to dominate rather than the surrounding overhead.

The confidence intervals also get smaller as file size increases, which makes sense, as larger files take longer to process, so the measurements are less affected by background system noise.

### Plot

![SHA Digests Generation](sha256_benchmark.png)

*Figure: SHA digests generation times (X-axis: bytes, Y-axis: $\mu$s)*

The linear plot shows that execution time scales roughly proportionally with file size for larger inputs, which is what we expect from a hash function processing data in fixed-size blocks. Meanwhile, in the log-log plot, the curve isn't a straight line, which tells us the relationship isn't cleanly linear across all scales. The steep rise at small file sizes followed by a gradually flattening slope reflects the shift from overhead-dominated to compute-dominated execution that the throughput table also shows. The error bars are only really visible on the mid-range file sizes, which lines up with what the CI column in the table already suggested.